In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit

In [ ]:
TASK = Path.cwd().parent.parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [ ]:
TAG = 'ar1_s'

In [ ]:
pairs = pd.read_csv(INPUT / 'transitions.csv', dtype={'stage':str})
panel = pd.read_csv(INPUT / 'panel.csv', index_col='dblp_id')
q0 = panel['0'].dropna().to_numpy()
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)

In [ ]:
rows = []
for label, start, end in CONFIG['fit_stages']:
    block = pairs.loc[pairs.age.between(start, end)]
    X = np.column_stack([np.ones(len(block)), block.z])
    coef = np.linalg.lstsq(X, block.z_next, rcond=None)[0]
    residual = block.z_next - X @ coef
    rows.append({'stage':label, 'start':start, 'end':end, 'intercept':coef[0], 'beta':coef[1],
                 'sigma':np.sqrt(np.mean(residual**2)), 'n':len(block)})
params = pd.DataFrame(rows)
params.to_csv(DEST / 'parameters.csv', index=False)
pd.DataFrame([{'scale':q0.mean(), 'n':len(q0)}]).to_csv(DEST / 'initialization.csv', index=False)
display(params)